# 04 — Representation Comparison & Model Selection

Compare three feature representations under proper cross-validation, with **all** learned transformations
(imputation, scaling, correlation pruning, feature selection, PCA, encoding) fit **inside each CV fold** on the
training fold only:

- **A — Existing PCA**: correlation-pruned connectome → PCA(0.95 variance)
- **B — Supervised Connectome Selection**: connectome → `SelectKBest(f_classif, k)` for k in {100, 250, 500}
- **C — Selected Connectome + Selected Metadata**: (B) + quantitative/categorical metadata

Two candidate models only (per project scope), chosen for this specific data regime (see justification below):
**Logistic Regression** (L2) and **XGBoost** (depth-constrained). All CV is 5-fold stratified on `ADHD_Outcome`,
computed on the TRAIN split only — validation and test subjects are never touched here.

In [1]:

import os, gc, time, warnings
warnings.filterwarnings('ignore')
os.chdir('/home/claude/adhd_project')
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

RANDOM_STATE = 123

fcm = pd.read_csv('data/processed/fcm_clean.csv')
quan = pd.read_csv('data/processed/quan_clean.csv')
cat = pd.read_csv('data/processed/cat_clean.csv')
target = pd.read_csv('data/processed/target_clean.csv')
ids_train = pd.read_csv('data/processed/ids_train.csv')['participant_id']

print('Full cleaned sets:', fcm.shape, quan.shape, cat.shape, target.shape)
print('N train subjects:', len(ids_train))


Full cleaned sets: (1101, 19901) (1101, 18) (1101, 3) (1101, 2)
N train subjects: 770


In [2]:

# Restrict to TRAIN split only for representation/model selection -- validation and test are untouched here.
fcm_tr = fcm[fcm['participant_id'].isin(ids_train)].sort_values('participant_id').reset_index(drop=True)
quan_tr = quan[quan['participant_id'].isin(ids_train)].sort_values('participant_id').reset_index(drop=True)
cat_tr = cat[cat['participant_id'].isin(ids_train)].sort_values('participant_id').reset_index(drop=True)
y_tr = target[target['participant_id'].isin(ids_train)].sort_values('participant_id').reset_index(drop=True)['ADHD_Outcome'].values

assert (fcm_tr['participant_id'].values == quan_tr['participant_id'].values).all()
assert (fcm_tr['participant_id'].values == cat_tr['participant_id'].values).all()
print('Train-only shapes -- fcm:', fcm_tr.shape, 'quan:', quan_tr.shape, 'cat:', cat_tr.shape, 'y:', y_tr.shape)
print('Train ADHD rate: %.3f' % y_tr.mean())


Train-only shapes -- fcm: (770, 19901) quan: (770, 18) cat: (770, 3) y: (770,)
Train ADHD rate: 0.688


## Connectome redundancy pruning (unsupervised, TRAIN-only, fit once)

This step does not use the label `y` at all (pure feature-feature Spearman redundancy, matching notebook 03's
recipe), and is fit only on the 770 TRAIN subjects — validation/test subjects never influence it. Reduces 19,900
edges down before the CV loop so PCA/SelectKBest inside each fold run in reasonable time on this single-CPU
environment.

In [3]:

conn_cols = [c for c in fcm_tr.columns if c != 'participant_id']
X_conn_tr_full = fcm_tr[conn_cols].astype('float32').to_numpy()

t0 = time.time()
ranks = pd.DataFrame(X_conn_tr_full).rank(axis=0).to_numpy(dtype=np.float32).copy()
ranks = ranks - ranks.mean(axis=0, keepdims=True)
ranks = ranks / (ranks.std(axis=0, keepdims=True) + 1e-8)
n = ranks.shape[0]

to_drop_idx = set()
block = 1000
n_cols = ranks.shape[1]
for start in range(0, n_cols, block):
    end = min(start + block, n_cols)
    cb = (ranks[:, start:end].T @ ranks) / n
    for li, gi in enumerate(range(start, end)):
        row = cb[li]
        hc = np.where(np.abs(row[gi+1:]) >= 0.7)[0] + gi + 1
        for h in hc:
            to_drop_idx.add(h)
    del cb
del ranks
gc.collect()

keep_idx = [i for i in range(n_cols) if i not in to_drop_idx]
conn_cols_pruned = [conn_cols[i] for i in keep_idx]
X_conn_tr = X_conn_tr_full[:, keep_idx]
print(f'Pruning finished in {time.time()-t0:.1f}s. Kept {len(conn_cols_pruned)}/{n_cols} connectome edges.')


Pruning finished in 8.5s. Kept 12931/19900 connectome edges.


## Metadata design matrix (raw values kept; imputation/scaling/encoding done inside each CV fold)

In [4]:

quan_cols = [c for c in quan_tr.columns if c != 'participant_id']
cat_cols = [c for c in cat_tr.columns if c != 'participant_id']

X_quan_tr = quan_tr[quan_cols].to_numpy()
X_cat_tr = cat_tr[cat_cols].astype(str).to_numpy()  # treat as categorical codes for OneHotEncoder

print('Quantitative metadata features:', quan_cols)
print('Categorical metadata features:', cat_cols)
print('X_conn_tr:', X_conn_tr.shape, 'X_quan_tr:', X_quan_tr.shape, 'X_cat_tr:', X_cat_tr.shape)


Quantitative metadata features: ['EHQ_EHQ_Total', 'ColorVision_CV_Score', 'APQ_P_APQ_P_CP', 'APQ_P_APQ_P_ID', 'APQ_P_APQ_P_INV', 'APQ_P_APQ_P_OPD', 'APQ_P_APQ_P_PM', 'APQ_P_APQ_P_PP', 'SDQ_SDQ_Conduct_Problems', 'SDQ_SDQ_Difficulties_Total', 'SDQ_SDQ_Emotional_Problems', 'SDQ_SDQ_Externalizing', 'SDQ_SDQ_Generating_Impact', 'SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Internalizing', 'SDQ_SDQ_Peer_Problems', 'SDQ_SDQ_Prosocial']
Categorical metadata features: ['PreInt_Demos_Fam_Child_Ethnicity', 'PreInt_Demos_Fam_Child_Race']
X_conn_tr: (770, 12931) X_quan_tr: (770, 17) X_cat_tr: (770, 2)


## Model selection rationale

- **N train subjects**: 770 — small relative to feature counts (hundreds to tens of thousands depending on representation) → classic **p ≳ n / p ≫ n** regime.
- **Notebook 03 finding**: an unconstrained, deep (depth-9) XGBoost on 800+ PCA components hit train AUC 1.0 / test AUC ~0.51 — severe overfitting at this sample size.
- **Class imbalance**: moderate (2.18:1), doesn't demand exotic resampling — `class_weight='balanced'` / `scale_pos_weight` is sufficient.
- **Connectome structure**: diffuse, weakly-individually-predictive edges (EDA: only ~7% of edges significant at p<0.05) — favors models that can aggregate many weak, correlated signals without over-committing to any one split, i.e. **regularized linear models**.

**Chosen candidates:**
1. **Logistic Regression (L2, `class_weight='balanced'`)** — PRIMARY. Well-suited to p≫n, weak diffuse signal, resistant to the overfitting we already measured.
2. **XGBoost, depth-constrained (max_depth ≤ 4) with strong regularization** — BACKUP. Retains capacity for nonlinear metadata×connectome interactions but explicitly constrained given the overfitting evidence, rather than left at depth 9.

In [5]:

def make_representation_A(k_pca_variance=0.95):
    """Existing PCA: connectome only, correlation-pruned then PCA."""
    pre = ColumnTransformer([
        ('conn', Pipeline([
            ('scale', StandardScaler(with_mean=True, with_std=False)),
            ('pca', PCA(n_components=k_pca_variance, svd_solver='full', random_state=RANDOM_STATE)),
        ]), slice(0, X_conn_tr.shape[1])),
    ])
    return pre

def make_representation_B(k):
    """Supervised connectome selection: SelectKBest(f_classif, k), connectome only."""
    pre = ColumnTransformer([
        ('conn', Pipeline([
            ('select', SelectKBest(f_classif, k=k)),
            ('scale', StandardScaler()),
        ]), slice(0, X_conn_tr.shape[1])),
    ])
    return pre

def make_representation_C(k):
    """Selected connectome (top-k) + metadata (quantitative scaled + categorical one-hot)."""
    n_conn = X_conn_tr.shape[1]
    n_quan = X_quan_tr.shape[1]
    pre = ColumnTransformer([
        ('conn', Pipeline([
            ('select', SelectKBest(f_classif, k=k)),
            ('scale', StandardScaler()),
        ]), slice(0, n_conn)),
        ('quan', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), slice(n_conn, n_conn + n_quan)),
        ('cat', OneHotEncoder(handle_unknown='ignore'), slice(n_conn + n_quan, n_conn + n_quan + X_cat_tr.shape[1])),
    ])
    return pre

print('Representation builders ready.')


Representation builders ready.


In [6]:

X_full_for_C = np.hstack([X_conn_tr, X_quan_tr, X_cat_tr])  # cat columns kept as strings; ColumnTransformer slices handle types
# OneHotEncoder needs a homogeneous string/object block -- build a combined frame instead for safety
X_conn_df = pd.DataFrame(X_conn_tr, columns=[f'c{i}' for i in range(X_conn_tr.shape[1])])
X_quan_df = pd.DataFrame(X_quan_tr, columns=quan_cols)
X_cat_df = pd.DataFrame(X_cat_tr, columns=cat_cols)
X_C_df = pd.concat([X_conn_df, X_quan_df, X_cat_df], axis=1)
n_conn = X_conn_tr.shape[1]
n_quan = X_quan_tr.shape[1]
n_cat = X_cat_tr.shape[1]
print('Combined C matrix shape:', X_C_df.shape, '(conn=%d, quan=%d, cat=%d)' % (n_conn, n_quan, n_cat))


Combined C matrix shape: (770, 12950) (conn=12931, quan=17, cat=2)


## Run the CV leaderboard
5-fold stratified CV, train-only. For B and C we scan k in {100, 250, 500} and keep the best k per representation (kept small per project scope: this is a limited scan, not an exhaustive search).

In [7]:

def get_models():
    logreg = LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
    xgb = XGBClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6,
        reg_alpha=0.5, reg_lambda=2.0, min_child_weight=5,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=1,
        scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum(),
    )
    return {'LogisticRegression': logreg, 'XGBoost_shallow': xgb}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'roc_auc', 'f1', 'balanced_accuracy']

results = []
t_all = time.time()

# --- Representation A: PCA connectome-only ---
Xa = pd.DataFrame(X_conn_tr, columns=[f'c{i}' for i in range(X_conn_tr.shape[1])])
pre_a = make_representation_A()
for mname, model in get_models().items():
    t0 = time.time()
    pipe = Pipeline([('pre', pre_a), ('clf', model)])
    cvres = cross_validate(pipe, Xa, y_tr, cv=cv, scoring=scoring, n_jobs=1)
    results.append({
        'representation': 'A: Existing PCA (connectome only)', 'model': mname, 'k': 'PCA(0.95var)',
        'cv_accuracy': cvres['test_accuracy'].mean(), 'cv_accuracy_std': cvres['test_accuracy'].std(),
        'cv_balanced_accuracy': cvres['test_balanced_accuracy'].mean(),
        'cv_f1': cvres['test_f1'].mean(), 'cv_roc_auc': cvres['test_roc_auc'].mean(),
        'cv_roc_auc_std': cvres['test_roc_auc'].std(),
    })
    print(f'A / {mname} done in {time.time()-t0:.1f}s -> ROC-AUC {results[-1]["cv_roc_auc"]:.4f}')

print('Representation A finished. Elapsed: %.1fs' % (time.time()-t_all))


A / LogisticRegression done in 7.7s -> ROC-AUC 0.5602


A / XGBoost_shallow done in 12.0s -> ROC-AUC 0.5377
Representation A finished. Elapsed: 19.9s


In [8]:

# --- Representation B: supervised connectome selection ---
for k in [100, 250, 500]:
    pre_b = make_representation_B(k)
    for mname, model in get_models().items():
        t0 = time.time()
        pipe = Pipeline([('pre', pre_b), ('clf', model)])
        cvres = cross_validate(pipe, Xa, y_tr, cv=cv, scoring=scoring, n_jobs=1)
        results.append({
            'representation': 'B: Supervised Connectome Selection', 'model': mname, 'k': k,
            'cv_accuracy': cvres['test_accuracy'].mean(), 'cv_accuracy_std': cvres['test_accuracy'].std(),
            'cv_balanced_accuracy': cvres['test_balanced_accuracy'].mean(),
            'cv_f1': cvres['test_f1'].mean(), 'cv_roc_auc': cvres['test_roc_auc'].mean(),
            'cv_roc_auc_std': cvres['test_roc_auc'].std(),
        })
        print(f'B k={k} / {mname} done in {time.time()-t0:.1f}s -> ROC-AUC {results[-1]["cv_roc_auc"]:.4f}')

print('Representation B finished. Elapsed: %.1fs' % (time.time()-t_all))


B k=100 / LogisticRegression done in 1.3s -> ROC-AUC 0.5723


B k=100 / XGBoost_shallow done in 2.3s -> ROC-AUC 0.5862


B k=250 / LogisticRegression done in 1.4s -> ROC-AUC 0.5157


B k=250 / XGBoost_shallow done in 4.0s -> ROC-AUC 0.5805


B k=500 / LogisticRegression done in 1.4s -> ROC-AUC 0.5657


B k=500 / XGBoost_shallow done in 6.6s -> ROC-AUC 0.5805
Representation B finished. Elapsed: 36.8s


In [9]:

# --- Representation C: selected connectome + metadata ---
for k in [100, 250, 500]:
    pre_c = make_representation_C(k)
    for mname, model in get_models().items():
        t0 = time.time()
        pipe = Pipeline([('pre', pre_c), ('clf', model)])
        cvres = cross_validate(pipe, X_C_df, y_tr, cv=cv, scoring=scoring, n_jobs=1)
        results.append({
            'representation': 'C: Selected Connectome + Metadata', 'model': mname, 'k': k,
            'cv_accuracy': cvres['test_accuracy'].mean(), 'cv_accuracy_std': cvres['test_accuracy'].std(),
            'cv_balanced_accuracy': cvres['test_balanced_accuracy'].mean(),
            'cv_f1': cvres['test_f1'].mean(), 'cv_roc_auc': cvres['test_roc_auc'].mean(),
            'cv_roc_auc_std': cvres['test_roc_auc'].std(),
        })
        print(f'C k={k} / {mname} done in {time.time()-t0:.1f}s -> ROC-AUC {results[-1]["cv_roc_auc"]:.4f}')

print('Representation C finished. Total elapsed: %.1fs' % (time.time()-t_all))


C k=100 / LogisticRegression done in 1.3s -> ROC-AUC 0.7551


C k=100 / XGBoost_shallow done in 2.3s -> ROC-AUC 0.8193


C k=250 / LogisticRegression done in 1.3s -> ROC-AUC 0.7132


C k=250 / XGBoost_shallow done in 3.8s -> ROC-AUC 0.8297


C k=500 / LogisticRegression done in 1.4s -> ROC-AUC 0.7238


C k=500 / XGBoost_shallow done in 6.3s -> ROC-AUC 0.8305
Representation C finished. Total elapsed: 53.2s


## Representation leaderboard

In [10]:

res_df = pd.DataFrame(results).sort_values('cv_roc_auc', ascending=False).reset_index(drop=True)
os.makedirs('reports', exist_ok=True)
res_df.to_csv('reports/representation_comparison.csv', index=False)
res_df


,representation,model,k,cv_accuracy,cv_accuracy_std,cv_balanced_accuracy,cv_f1,cv_roc_auc,cv_roc_auc_std
0,C: Selected Connectome + Metadata,XGBoost_shallow,500,0.779221,0.018366,0.740448,0.840145,0.830542,0.018333
1,C: Selected Connectome + Metadata,XGBoost_shallow,250,0.777922,0.018089,0.739505,0.839068,0.829717,0.022116
2,C: Selected Connectome + Metadata,XGBoost_shallow,100,0.772727,0.012321,0.739151,0.833853,0.819261,0.010427
3,C: Selected Connectome + Metadata,LogisticRegression,100,0.712987,0.033465,0.676376,0.786818,0.755149,0.021174
4,C: Selected Connectome + Metadata,LogisticRegression,500,0.701299,0.007113,0.645086,0.785375,0.723821,0.003492
5,C: Selected Connectome + Metadata,LogisticRegression,250,0.696104,0.015584,0.649292,0.777794,0.713247,0.005673
6,B: Supervised Connectome Selection,XGBoost_shallow,100,0.631169,0.037952,0.557665,0.736983,0.586203,0.045363
7,B: Supervised Connectome Selection,XGBoost_shallow,500,0.633766,0.036178,0.540173,0.747316,0.580503,0.044313
8,B: Supervised Connectome Selection,XGBoost_shallow,250,0.627273,0.021181,0.543436,0.738528,0.580464,0.038361
9,B: Supervised Connectome Selection,LogisticRegression,100,0.609091,0.057113,0.561006,0.707018,0.572288,0.051892


In [11]:

best_row = res_df.iloc[0]
print('WINNER')
print('='*60)
print('Representation:', best_row['representation'])
print('Model:', best_row['model'])
print('k:', best_row['k'])
print('CV ROC-AUC: %.4f +/- %.4f' % (best_row['cv_roc_auc'], best_row['cv_roc_auc_std']))
print('CV Accuracy: %.4f +/- %.4f' % (best_row['cv_accuracy'], best_row['cv_accuracy_std']))
print('CV Balanced Accuracy: %.4f' % best_row['cv_balanced_accuracy'])
print('CV F1: %.4f' % best_row['cv_f1'])


WINNER
Representation: C: Selected Connectome + Metadata
Model: XGBoost_shallow
k: 500
CV ROC-AUC: 0.8305 +/- 0.0183
CV Accuracy: 0.7792 +/- 0.0184
CV Balanced Accuracy: 0.7404
CV F1: 0.8401


In [12]:

import json
os.makedirs('artifacts', exist_ok=True)
winner_config = {
    'representation': str(best_row['representation']),
    'model': str(best_row['model']),
    'k': None if best_row['k'] == 'PCA(0.95var)' else int(best_row['k']),
    'use_pca': bool(best_row['k'] == 'PCA(0.95var)'),
    'cv_roc_auc_mean': float(best_row['cv_roc_auc']),
    'cv_roc_auc_std': float(best_row['cv_roc_auc_std']),
}
with open('artifacts/winner_config.json', 'w') as f:
    json.dump(winner_config, f, indent=2)
print(winner_config)


{'representation': 'C: Selected Connectome + Metadata', 'model': 'XGBoost_shallow', 'k': 500, 'use_pca': False, 'cv_roc_auc_mean': 0.8305424528301888, 'cv_roc_auc_std': 0.01833270290834914}
